# Quark Orbital ZBW DP Mixing Fractions (Up Quark + N_k Sensitivity)

This notebook reproduces the quark orbital ZBW composition (~74% qDP) for up quark (N_k=1) and shows N_k dependence.

## 1. Imports and Constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

phi = (1 + np.sqrt(5)) / 2
base_gradient = 1.0
gradients = {
    'qDP': (phi - 1) * base_gradient,  # favorable for quarks
    'hDP': base_gradient,
    'eDP': phi * base_gradient           # least favorable
}

## 2. Function Definitions (same as lepton notebook)

In [ ]:
def energy(f):
    f_q, f_h, f_e = f
    E = f_q * gradients['qDP'] + f_h * gradients['hDP'] + f_e * gradients['eDP']
    return E

cons = ({'type': 'eq', 'fun': lambda f: np.sum(f) - 1})
bounds = [(0, 1)] * 3

## 3. Single Calculation (Up Quark, N_k=1)

In [ ]:
N_k = 1
T = N_k
res = minimize(energy, [1/3, 1/3, 1/3], method='SLSQP', bounds=bounds, constraints=cons)
min_fractions = res.x
probs = np.exp(-np.array([gradients['qDP'], gradients['hDP'], gradients['eDP']]) / T)
probs /= np.sum(probs)
equil = 0.5 * min_fractions + 0.5 * probs
equil /= np.sum(equil)

print("Up quark (N_k=1) equilibrium fractions:", equil)

## 4. N_k Sensitivity Sweep

In [ ]:
N_k_values = [1, 4, 12, 20, 60, 30000]  # up → top
qDP_fracs = []
for nk in N_k_values:
    T = nk
    probs = np.exp(-np.array([gradients['qDP'], gradients['hDP'], gradients['eDP']]) / T)
    probs /= np.sum(probs)
    # Simplified: use thermal only for sweep (min is always qDP-heavy)
    qDP_fracs.append(probs[0])

plt.plot(N_k_values, qDP_fracs, marker='o')
plt.xscale('log')
plt.xlabel('N_k (cage occupancy)')
plt.ylabel('qDP fraction in orbital ZBW')
plt.title('Quark Orbital ZBW: qDP Dominance vs. Cage Size')
plt.grid(True, which="both", ls="--")
plt.savefig('../figures/sensitivity_plot.png')
plt.show()